In [143]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import numpy as np
import requests
import yaml

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

CONFIG_PATH = Path('config.yaml')
if not CONFIG_PATH.exists():
    raise FileNotFoundError('config.yaml not found in current folder.')

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f) or {}

# Load model config dynamically from active_model setting
ACTIVE_MODEL = str(CONFIG.get('active_model', 'phi4')).strip().lower()
models_config = CONFIG.get('models', {})
if ACTIVE_MODEL in models_config:
    MODEL = models_config.get(ACTIVE_MODEL, {})
else:
    MODEL = CONFIG.get('model', {})

PROVIDER = str(MODEL.get('provider', 'vllm')).strip().lower()
MODEL_NAME = str(MODEL.get('name', '')).strip()
BASE_URL = str(MODEL.get('base_url', '')).strip()
MODEL_DESCRIPTION = str(MODEL.get('description', '')).strip()
TOP_K = int(CONFIG.get('top_k', 5))
PROPERTIES_TO_EXTRACT = CONFIG.get(
    'properties',
    ['license', 'installation', 'contact', 'contributors', 'links', 'description']
)

repositories = CONFIG.get('repositories', [])
ENABLED_REPOS = [r for r in repositories if isinstance(r, dict) and r.get('enabled', True) and r.get('url')]

print('Loaded config from:', CONFIG_PATH.resolve())
print(f'Active model: {ACTIVE_MODEL}')
print(f'Provider: {PROVIDER}')
print(f'Model name: {MODEL_NAME}')
print(f'Description: {MODEL_DESCRIPTION}')
print(f'Base URL: {BASE_URL}')
print(f'Top K: {TOP_K}')
print(f'Enabled repositories: {len(ENABLED_REPOS)}')

Loaded config from: /home/ubuntu/comet-rs/maSMP-metadata-extraction/config.yaml
Active model: gemma4
Provider: vllm
Model name: google/gemma-4-E4B-it
Description: Fastest Latency. Good for instant UIs.
Base URL: http://localhost:8000
Top K: 5
Enabled repositories: 1


In [144]:
PROPERTY_QUERIES = {
    'license': ['license', 'licensing', 'copyright', 'spdx'],
    'installation': ['installation', 'install', 'pip', 'conda', 'requirements', 'setup'],
    'contact': ['contact', 'email', 'maintainer', 'author'],
    'contributors': ['contributors', 'contributor', 'authors', 'maintainers', 'team', 'credits', '@', 'github.com'],
    'links': ['paper', 'publication', 'citation', 'arxiv', 'doi', 'docs', 'link', 'url'],
    'description': ['overview', 'about', 'purpose', 'project', 'toolkit', 'library', 'framework', 'features', 'introduction']
}

PROPERTY_SCHEMA_HINTS = {
    'license': 'normalized SPDX-style string if explicit, else null',
    'installation': 'short installation instruction summary, else null',
    'contact': 'email or contact link, else null',
    'contributors': 'list of contributors with name/github_url if available, else null',
    'links': 'list of relevant links as objects {title, url, relevance, is_working, status_code}, else null',
    'description': 'one or two concise lines describing what the repository does, else null'
}

PROPERTY_RULES = {
    'license': 'Return normalized SPDX license and exact evidence quote when possible.',
    'contributors': 'Return value as list of objects [{\"name\": string, \"github_url\": string|null}].',
    'links': 'Return value as list of objects [{\"title\": string|null, \"url\": string, \"relevance\": \"paper\"|\"docs\"|\"repo\"|\"tutorial\"|\"other\", \"is_working\": bool|null, \"status_code\": int|null}].',
    'description': 'Return factual concise repository description from evidence only.',
    'default': 'Return concise value and evidence quote if available.'
}

LICENSE_PATTERNS = [
    (r'bsd\s*[- ]?3\s*[- ]?clause|bsd-3-clause', 'BSD-3-Clause'),
    (r'bsd\s*[- ]?2\s*[- ]?clause|bsd-2-clause', 'BSD-2-Clause'),
    (r'\bmit\b(?:\s+license)?', 'MIT'),
    (r'apache\s*2\.0|apache-2\.0|apache license', 'Apache-2.0'),
    (r'gpl\s*v?3|gnu\s+general\s+public\s+license\s*v?3', 'GPL-3.0'),
    (r'gpl\s*v?2|gnu\s+general\s+public\s+license\s*v?2', 'GPL-2.0'),
    (r'lgpl\s*v?3|lesser\s+general\s+public\s+license\s*v?3', 'LGPL-3.0'),
    (r'mpl\s*2\.0|mozilla\s+public\s+license\s*2\.0', 'MPL-2.0')
]

@dataclass
class RepoRef:
    provider: str
    owner: str
    repo: str

def parse_repo_url(repo_url: str) -> RepoRef:
    """Parse a Git repository URL into provider, owner, and repository name.

    Args:
        repo_url: Full repository URL from GitHub or GitLab.

    Returns:
        A `RepoRef` object with normalized provider, owner, and repo.
    """
    parsed = urlparse(repo_url)
    host = parsed.netloc.lower()
    path_parts = [p for p in parsed.path.strip('/').split('/') if p]
    if len(path_parts) < 2:
        raise ValueError(f'Invalid repository URL: {repo_url}')

    owner, repo = path_parts[0], path_parts[1]
    if repo.endswith('.git'):
        repo = repo[:-4]

    if 'github.com' in host:
        provider = 'github'
    elif 'gitlab.com' in host:
        provider = 'gitlab'
    else:
        raise ValueError('Only GitHub and GitLab URLs are supported.')

    return RepoRef(provider=provider, owner=owner, repo=repo)

def candidate_readme_urls(ref: RepoRef) -> list[str]:
    """Generate candidate raw README URLs for common branches and file names.

    Args:
        ref: Parsed repository reference.

    Returns:
        A list of candidate README URLs to probe in order.
    """
    branches = ['main', 'master']
    filenames = ['README.md', 'README.MD', 'readme.md', 'README.rst', 'README.txt']
    urls = []
    for branch in branches:
        for filename in filenames:
            if ref.provider == 'github':
                urls.append(f'https://raw.githubusercontent.com/{ref.owner}/{ref.repo}/{branch}/{filename}')
            else:
                urls.append(f'https://gitlab.com/{ref.owner}/{ref.repo}/-/raw/{branch}/{filename}')
    return urls

def fetch_readme(ref: RepoRef, timeout: int = 20) -> tuple[str, str]:
    """Fetch README content from the first reachable candidate URL.

    Args:
        ref: Parsed repository reference.
        timeout: HTTP timeout in seconds for each request.

    Returns:
        A tuple `(readme_text, source_url)` for the resolved README.
    """
    headers = {'User-Agent': 'maSMP-generic-readme-extraction/1.0'}
    for url in candidate_readme_urls(ref):
        try:
            resp = requests.get(url, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp.text, url
        except Exception:
            continue
    raise RuntimeError('README not found on main/master with common filenames.')

def split_with_metadata(md_text: str) -> list[dict]:
    """Split markdown text into sections while preserving heading metadata.

    Args:
        md_text: Raw markdown text.

    Returns:
        A list of section dictionaries containing heading, level, and content.
    """
    lines = md_text.split('\n')
    chunks = []
    current = {'heading': None, 'level': None, 'content': []}

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        m = re.match(r'^(#{1,6})\s+(.*)', line)
        if m:
            if current['content']:
                chunks.append(current)
            current = {'heading': m.group(2), 'level': len(m.group(1)), 'content': []}
            i += 1
            continue

        if i + 1 < len(lines):
            next_line = lines[i + 1].strip()
            if re.match(r'^=+$', next_line):
                if current['content']:
                    chunks.append(current)
                current = {'heading': line, 'level': 1, 'content': []}
                i += 2
                continue
            if re.match(r'^-+$', next_line):
                if current['content']:
                    chunks.append(current)
                current = {'heading': line, 'level': 2, 'content': []}
                i += 2
                continue

        current['content'].append(lines[i])
        i += 1

    if current['content']:
        chunks.append(current)

    return chunks

def hybrid_chunking(section: dict, max_chars: int = 1200, overlap: int = 200) -> list[dict]:
    """Chunk a section into overlapping text windows when it is too large.

    Args:
        section: Section dictionary with heading and content.
        max_chars: Maximum number of characters per chunk.
        overlap: Character overlap between adjacent chunks.

    Returns:
        A list of chunk dictionaries with heading and chunked content.
    """
    text = '\n'.join(section['content']).strip()
    if len(text) <= max_chars:
        return [{'heading': section['heading'], 'content': text}]

    out = []
    start = 0
    while start < len(text):
        end = start + max_chars
        out.append({'heading': section['heading'], 'content': text[start:end]})
        start += max_chars - overlap
    return out

def prepare_chunk_records(chunks: list[dict]) -> list[dict]:
    """Normalize chunk dictionaries into retrieval-ready records.

    Args:
        chunks: Raw chunk dictionaries.

    Returns:
        A list of normalized records with `full_text` and metadata.
    """
    records = []
    for i, c in enumerate(chunks):
        heading = '' if c.get('heading') is None else str(c.get('heading')).strip()
        content = '' if c.get('content') is None else str(c.get('content')).strip()
        if not content:
            continue
        records.append({
            'chunk_id': i,
            'heading': heading,
            'content': content,
            'full_text': f'Heading: {heading}\n\n{content}' if heading else content
        })
    return records

def build_retrieval_index(records: list[dict], model_name: str = 'sentence-transformers/all-MiniLM-L6-v2') -> dict:
    """Build lexical and optional embedding index for chunk retrieval.

    Args:
        records: Normalized chunk records.
        model_name: Sentence-transformers model name for embeddings.

    Returns:
        Retrieval index dictionary with records and optional embeddings.
    """
    index = {
        'records': records,
        'embeddings': None,
        'model': None,
        'embedding_enabled': False
    }

    if SentenceTransformer is None:
        print('[Embeddings] disabled: sentence-transformers not available')
        return index

    try:
        model = SentenceTransformer(model_name)
        vec = model.encode([r['full_text'] for r in records], normalize_embeddings=True)
        index['embeddings'] = np.asarray(vec, dtype=np.float32)
        index['model'] = model
        index['embedding_enabled'] = True
        print(f'[Embeddings] enabled ({model_name})')
    except Exception as exc:
        print('[Embeddings] disabled, lexical fallback only:', exc)

    return index

def keyword_score(record: dict, query_terms: list[str]) -> float:
    """Compute simple lexical relevance score for one chunk record.

    Args:
        record: One retrieval record containing heading and content.
        query_terms: Property-specific query terms.

    Returns:
        A non-negative lexical relevance score.
    """
    heading = record['heading'].lower()
    content = record['content'].lower()
    score = 0.0
    for term in query_terms:
        t = term.lower()
        if t in heading:
            score += 2.0
        if t in content:
            score += 1.0
    return score

def retrieve_top_chunks(index: dict, property_name: str, top_k: int = 5, alpha: float = 0.75) -> list[dict]:
    """Retrieve top chunks using hybrid semantic and lexical scoring.

    Args:
        index: Retrieval index created by `build_retrieval_index`.
        property_name: Target property key used to select query terms.
        top_k: Number of top chunks to return.
        alpha: Weight for semantic score in hybrid mode.

    Returns:
        Ranked list of top chunk records with score and rank fields.
    """
    terms = PROPERTY_QUERIES[property_name]
    records = index['records']

    kw = np.array([keyword_score(r, terms) for r in records], dtype=np.float32)
    if kw.size and kw.max() > 0:
        kw = kw / kw.max()

    if index['embedding_enabled']:
        q = ' '.join(terms)
        qv = index['model'].encode([q], normalize_embeddings=True)[0]
        sem = index['embeddings'] @ np.asarray(qv, dtype=np.float32)
        sem = (sem + 1.0) / 2.0
        scores = alpha * sem + (1 - alpha) * kw
    else:
        scores = kw

    order = np.argsort(-scores)[:top_k]
    out = []
    for rank, idx in enumerate(order, start=1):
        r = dict(records[int(idx)])
        r['score'] = float(scores[int(idx)])
        r['rank'] = rank
        out.append(r)
    return out

In [145]:
def extract_links_from_text(text: str, check_health: bool = False, health_timeout: int = 3) -> list[dict]:
    """Extract markdown and bare HTTP(S) links from free text.

    Args:
        text: Input text (typically README content).

    Returns:
        A de-duplicated list of link objects with `title`, `url`, `relevance`,
        `is_working`, and `status_code`.
    """
    found = []
    seen = set()

    # Markdown links: [title](url)
    for title, url in re.findall(r'\[([^\]]+)\]\((https?://[^\s)]+)\)', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen:
            seen.add(u)
            found.append({'title': title.strip(), 'url': u})

    # Bare URLs
    for url in re.findall(r'https?://[^\s<>()\]\[\]"\'`]+', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen:
            seen.add(u)
            found.append({'title': None, 'url': u})

    for item in found:
        u = (item.get('url') or '').lower()
        t = (item.get('title') or '').lower()
        if any(k in u or k in t for k in ['arxiv', 'doi.org', 'paper', 'publication', 'proceedings', '.pdf']):
            item['relevance'] = 'paper'
        elif any(k in u or k in t for k in ['docs', 'documentation', 'readthedocs', 'gitbook', 'wiki', 'guide']):
            item['relevance'] = 'docs'
        elif any(k in u or k in t for k in ['github', 'gitlab', 'bitbucket']):
            item['relevance'] = 'repo'
        elif any(k in u or k in t for k in ['example', 'demo', 'tutorial', 'howto']):
            item['relevance'] = 'tutorial'
        else:
            item['relevance'] = 'other'

        if check_health:
            try:
                r = requests.head(item['url'], timeout=health_timeout, allow_redirects=True)
                item['status_code'] = int(r.status_code)
                item['is_working'] = 200 <= r.status_code < 400
            except Exception:
                item['status_code'] = None
                item['is_working'] = False
        else:
            item['status_code'] = None
            item['is_working'] = None

    return found

In [146]:
def check_provider_ready(provider: str, model: str, base_url: str, timeout: int = 10) -> tuple[bool, str]:
    """Check whether the configured inference provider and model are reachable.

    Args:
        provider: Backend provider name (`ollama` or `vllm`).
        model: Model identifier expected to be available on the backend.
        base_url: Base URL of the backend server.
        timeout: HTTP timeout in seconds.

    Returns:
        Tuple `(is_ready, message)` with readiness status and details.
    """
    provider = provider.lower().strip()

    if provider == 'ollama':
        endpoint = base_url.rstrip('/') + '/api/tags'
        try:
            resp = requests.get(endpoint, timeout=timeout)
            resp.raise_for_status()
            names = [m.get('name', '') for m in resp.json().get('models', [])]
            if model in names:
                return True, f'Ollama ready. Model {model} found.'
            return False, f'Model {model} not found. Available: {names}'
        except Exception as exc:
            return False, f'Cannot reach Ollama: {exc}'

    if provider == 'vllm':
        endpoint = base_url.rstrip('/') + '/v1/models'
        try:
            resp = requests.get(endpoint, timeout=timeout)
            resp.raise_for_status()
            names = [m.get('id', '') for m in resp.json().get('data', [])]
            if model in names:
                return True, f'vLLM ready. Model {model} found.'
            return False, f'Model {model} not found. Available: {names}'
        except Exception as exc:
            return False, f'Cannot reach vLLM: {exc}'

    return False, f'Unsupported provider: {provider}'

def run_llm(prompt: str, provider: str, model: str, base_url: str, timeout: int = 420) -> str:
    """Send a prompt to the selected backend and return model text output.

    Args:
        prompt: Prompt text to send.
        provider: Backend provider name (`ollama` or `vllm`).
        model: Model identifier to use.
        base_url: Base URL of the backend server.
        timeout: Request timeout in seconds.

    Returns:
        Raw response text from the model.
    """
    provider = provider.lower().strip()

    if provider == 'ollama':
        endpoint = base_url.rstrip('/') + '/api/generate'
        payload = {
            'model': model,
            'prompt': prompt,
            'stream': False,
            'format': 'json',
            'options': {'temperature': 0, 'top_p': 0.7, 'num_predict': 700},
        }
        resp = requests.post(endpoint, json=payload, timeout=timeout)
        resp.raise_for_status()
        return resp.json().get('response', '')

    if provider == 'vllm':
        endpoint = base_url.rstrip('/') + '/v1/chat/completions'
        payload = {
            'model': model,
            'messages': [
                {'role': 'system', 'content': 'Return valid JSON only.'},
                {'role': 'user', 'content': prompt},
            ],
            'temperature': 0,
            'top_p': 0.7,
            'max_tokens': 2000,
        }
        resp = requests.post(endpoint, json=payload, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        return data['choices'][0]['message']['content']

    raise RuntimeError(f'Unsupported provider: {provider}')

def extract_json(text: str) -> dict:
    """Parse JSON from model output, with fallback extraction from text body.

    Args:
        text: Raw model output string.

    Returns:
        Parsed JSON dictionary or a default empty-result dictionary.
    """
    text = (text or '').strip()
    if not text:
        return {'value': None, 'evidence': None, 'confidence': 0.0}

    # Strip fenced code blocks if the model wrapped JSON in markdown.
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE).strip()
    text = re.sub(r'\s*```$', '', text).strip()

    def _normalize(parsed: Any) -> dict:
        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, list):
            for x in parsed:
                if isinstance(x, dict):
                    return x
        return {'value': None, 'evidence': None, 'confidence': 0.0}

    try:
        return _normalize(json.loads(text))
    except Exception:
        pass

    # Try decoding the first JSON value in noisy output.
    try:
        decoder = json.JSONDecoder()
        parsed, _ = decoder.raw_decode(text)
        return _normalize(parsed)
    except Exception:
        pass

    # Fallback: first JSON object region.
    m = re.search(r'\{[\s\S]*?\}', text)
    if m:
        try:
            return _normalize(json.loads(m.group(0)))
        except Exception:
            pass

    return {'value': None, 'evidence': None, 'confidence': 0.0}

if 'extract_license_from_readme' not in globals():
    def extract_license_from_readme(readme_text: str) -> tuple[str | None, str | None]:
        """Detect a normalized SPDX-like license and evidence from README text.

        Args:
            readme_text: Full README markdown text.

        Returns:
            Tuple `(license_value, evidence)` or `(None, None)` if not found.
        """
        low = readme_text.lower()
        for pat, spdx in LICENSE_PATTERNS:
            m = re.search(pat, low, flags=re.IGNORECASE)
            if m:
                start = max(0, m.start() - 80)
                end = min(len(readme_text), m.end() + 80)
                evidence = readme_text[start:end].replace('\n', ' ').strip()
                return spdx, evidence
        return None, None

if 'extract_contributors_from_text' not in globals():
    def extract_contributors_from_text(text: str) -> list[dict]:
        """Extract contributor handles and GitHub profile links from text.

        Args:
            text: Source text where contributor references may appear.

        Returns:
            List of contributor objects with `name` and `github_url`.
        """
        found = []
        seen = set()

        for name, url in re.findall(r'\[([^\]]+)\]\((https?://(?:www\.)?github\.com/[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)?)\)', text, flags=re.IGNORECASE):
            u = url.strip()
            if u and u not in seen:
                seen.add(u)
                found.append({'name': name.strip(), 'github_url': u})

        for url in re.findall(r'https?://(?:www\.)?github\.com/[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)?', text, flags=re.IGNORECASE):
            u = url.strip().rstrip(').,;')
            if u and u not in seen:
                seen.add(u)
                found.append({'name': u.rstrip('/').split('/')[-1], 'github_url': u})

        for handle in re.findall(r'(?<![\w/])@([A-Za-z0-9-]{1,39})\b', text):
            u = f'https://github.com/{handle}'
            if u not in seen:
                seen.add(u)
                found.append({'name': handle, 'github_url': u})

        return found

def check_url_health(url: str, timeout: int = 5) -> tuple[bool, int | None]:
    """Check if a URL is reachable and return HTTP status code.

    Args:
        url: URL to check.
        timeout: HTTP timeout in seconds.

    Returns:
        Tuple (is_working, status_code). is_working=True if 2xx/3xx, status_code=None on error.
    """
    try:
        resp = requests.head(url, timeout=timeout, allow_redirects=True)
        is_working = 200 <= resp.status_code < 400
        return is_working, resp.status_code
    except Exception:
        try:
            resp = requests.get(url, timeout=timeout, allow_redirects=True, stream=True)
            resp.close()
            is_working = 200 <= resp.status_code < 400
            return is_working, resp.status_code
        except Exception:
            return False, None

if 'extract_links_from_text' not in globals():
    def extract_links_from_text(text: str, check_health: bool = True, health_timeout: int = 3) -> list[dict]:
        """Extract links from text with health checks and improved relevance detection.

        Args:
            text: Source text where URLs may appear.
            check_health: Whether to validate URLs (HEAD/GET requests).
            health_timeout: Timeout for health checks in seconds.

        Returns:
            List of link objects with title, url, relevance, is_working, status_code.
        """
        found = []
        seen = set()
        
        # Markdown links: [title](url)
        for title, url in re.findall(r'\[([^\]]+)\]\((https?://[^\s)]+)\)', text, flags=re.IGNORECASE):
            u = url.strip().rstrip(').,;')
            if u and u not in seen:
                seen.add(u)
                found.append({'title': title.strip(), 'url': u})
        
        # ReStructuredText links: `title <url>`_
        for title, url in re.findall(r'`([^<>`]+)\s*<(https?://[^>]+)>`_', text, flags=re.IGNORECASE):
            u = url.strip()
            if u and u not in seen:
                seen.add(u)
                found.append({'title': title.strip(), 'url': u})
        
        # Bare URLs
        for url in re.findall(r'https?://[^\s<>()\]\[\]"\'`]+', text, flags=re.IGNORECASE):
            u = url.strip().rstrip(').,;')
            if u and u not in seen:
                seen.add(u)
                found.append({'title': None, 'url': u})
        
        # Detect relevance for each link
        for item in found:
            url = item.get('url', '').lower()
            title = (item.get('title') or '').lower()
            
            # Categorize by relevance
            if any(k in url or k in title for k in ['arxiv', 'doi.org', 'paper', 'publication', 'proceedings', '.pdf']):
                item['relevance'] = 'paper'
            elif any(k in url or k in title for k in ['docs', 'documentation', 'readthedocs', 'gitbook', 'wiki', 'guide']):
                item['relevance'] = 'docs'
            elif any(k in url or k in title for k in ['github', 'gitlab', 'bitbucket']):
                item['relevance'] = 'repo'
            elif any(k in url or k in title for k in ['example', 'demo', 'tutorial', 'guide', 'howto']):
                item['relevance'] = 'tutorial'
            else:
                item['relevance'] = 'other'
            
            # Check health if enabled
            if check_health:
                is_working, status_code = check_url_health(item['url'], timeout=health_timeout)
                item['is_working'] = is_working
                item['status_code'] = status_code
            else:
                item['is_working'] = None
                item['status_code'] = None
        
        return found

def build_prompt(property_name: str, context: str) -> str:
    """Build a constrained extraction prompt for a single target property.

    Args:
        property_name: Name of the property to extract.
        context: Ranked chunk context provided to the model.

    Returns:
        Prompt string instructing the model to return strict JSON.
    """
    schema_hint = PROPERTY_SCHEMA_HINTS.get(property_name, 'null if unknown')
    rule_hint = PROPERTY_RULES.get(property_name, PROPERTY_RULES['default'])
    return (
        f"You are extracting '{property_name}' from repository README chunks.\n"
        f"Rules: {rule_hint}\n"
        f"Expected value shape: {schema_hint}\n\n"
        "Return JSON only with keys: value, evidence, confidence.\n"
        "- value: extracted value or null\n"
        "- evidence: exact short quote from context or null\n"
        "- confidence: number from 0 to 1\n\n"
        f"Context:\n{context}"
    )

def extract_property(property_name: str, index: dict, readme_text: str, provider: str, model: str, base_url: str, top_k: int = 5) -> dict:
    """Extract one property from README using heuristics plus LLM fallback.

    Args:
        property_name: Property key to extract.
        index: Retrieval index over README chunks.
        readme_text: Full README text.
        provider: Backend provider name.
        model: Model identifier.
        base_url: Backend base URL.
        top_k: Number of chunks to provide as LLM context.

    Returns:
        Dictionary with keys `value`, `evidence`, and `confidence`.
    """
    if property_name == 'license':
        lic, ev = extract_license_from_readme(readme_text)
        if lic:
            return {'value': lic, 'evidence': ev, 'confidence': 0.98}

    if property_name == 'contributors':
        c = extract_contributors_from_text(readme_text)
        if c:
            return {'value': c, 'evidence': 'Found GitHub handles/links in README.', 'confidence': 0.95}

    if property_name == 'links':
        # Extract links without health checks (set check_health=True for validation)
        links = extract_links_from_text(readme_text, check_health=False)
        if links:
            return {'value': links, 'evidence': 'Parsed links directly from README text.', 'confidence': 0.9}

    top = retrieve_top_chunks(index, property_name, top_k=top_k)
    context = '\n\n'.join([f"[Rank {c['rank']}, score={c['score']:.3f}] {c['full_text']}" for c in top])
    prompt = build_prompt(property_name, context)
    raw = run_llm(prompt, provider=provider, model=model, base_url=base_url)
    data = extract_json(raw)
    data.setdefault('value', None)
    data.setdefault('evidence', None)
    data.setdefault('confidence', 0.0)
    return data

def process_repository(repo_url: str, provider: str, model: str, base_url: str, top_k: int, properties: list[str]) -> dict:
    """Run complete extraction pipeline for one repository URL.

    Args:
        repo_url: Repository URL to process.
        provider: Backend provider name.
        model: Model identifier.
        base_url: Backend base URL.
        top_k: Number of chunks retrieved for each property.
        properties: List of property names to extract.

    Returns:
        Repository-level extraction result dictionary.
    """
    ref = parse_repo_url(repo_url)
    readme_text, readme_url = fetch_readme(ref)
    sections = split_with_metadata(readme_text)
    chunks = []
    for s in sections:
        chunks.extend(hybrid_chunking(s))
    records = prepare_chunk_records(chunks)
    index = build_retrieval_index(records)

    result = {
        'repo_url': repo_url,
        'readme_url': readme_url,
        'properties': {},
    }

    for p in properties:
        try:
            result['properties'][p] = extract_property(
                property_name=p,
                index=index,
                readme_text=readme_text,
                provider=provider,
                model=model,
                base_url=base_url,
                top_k=top_k,
            )
        except Exception as exc:
            result['properties'][p] = {
                'value': None,
                'evidence': None,
                'confidence': 0.0,
                'error': str(exc),
            }
    return result

ok, msg = check_provider_ready(PROVIDER, MODEL_NAME, BASE_URL, timeout=10)
print(msg)
if not ok:
    raise RuntimeError('Provider/model not ready.')

all_results = {}
for repo in ENABLED_REPOS:
    repo_url = str(repo.get('url')).strip()
    name = str(repo.get('name') or repo_url)
    print(f'\n=== Processing {name} ===')
    all_results[repo_url] = process_repository(
        repo_url=repo_url,
        provider=PROVIDER,
        model=MODEL_NAME,
        base_url=BASE_URL,
        top_k=TOP_K,
        properties=PROPERTIES_TO_EXTRACT,
    )

print('\nDone. Repo count:', len(all_results))
print(json.dumps(all_results, indent=2, ensure_ascii=False)[:4000])

vLLM ready. Model google/gemma-4-E4B-it found.

=== Processing dEFEND-web ===
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'

Done. Repo count: 1
{
  "https://github.com/cuilimeng/dEFEND-web.git": {
    "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
    "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
    "properties": {
      "license": {
        "value": null,
        "evidence": null,
        "confidence": 0.0
      },
      "installation": {
        "value": "Run the application using 'python application.py' after replacing the API key.",
        "evidence": "python application.py",
        "confidence": 0.9
      },
      "contact": {
        "value": null,
        "evidence": null,
        "confidence": 1.0
      },
      "contributors": {
        "value": [
          {
            "name": "inproceedings",
            "github_url": "https://github.com/inproceedings"
          }
  

In [147]:
out_path = Path('generic_extraction_results.json')
out_path.write_text(json.dumps(all_results, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', out_path.resolve())

Saved: /home/ubuntu/comet-rs/maSMP-metadata-extraction/generic_extraction_results.json


## Single-property test helper

Use this section to test extraction of exactly one property for one repository, without running the full batch loop.

In [148]:
from time import perf_counter

def timed_call(func, *args, **kwargs):
    """Run a function and return both result and elapsed seconds."""
    start = perf_counter()
    result = func(*args, **kwargs)
    elapsed = perf_counter() - start
    return result, elapsed

# Configure a single-property test
TEST_REPO_URL = (
    ENABLED_REPOS[0]['url']
    if ENABLED_REPOS
    else 'https://github.com/CERN/TIGRE'
 )
TEST_PROPERTY = 'license'  # e.g. 'installation', 'contact', 'contributors', 'links', 'description'

print('Testing repo:', TEST_REPO_URL)
print('Testing property:', TEST_PROPERTY)

single_property_result, elapsed_seconds = timed_call(
    process_repository,
    repo_url=TEST_REPO_URL,
    provider=PROVIDER,
    model=MODEL_NAME,
    base_url=BASE_URL,
    top_k=TOP_K,
    properties=[TEST_PROPERTY],
)

print(f'Elapsed time: {elapsed_seconds:.3f} seconds')
print(json.dumps(single_property_result, indent=2, ensure_ascii=False)[:4000])

Testing repo: https://github.com/cuilimeng/dEFEND-web.git
Testing property: license
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'
Elapsed time: 2.153 seconds
{
  "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
  "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
  "properties": {
    "license": {
      "value": null,
      "evidence": null,
      "confidence": 0.0
    }
  }
}


## Individual property test cells

Run any cell below to test one specific property independently.

In [149]:
 # Shared helper for individual property tests
def run_property_test(property_name: str, repo_url: str | None = None):
    target_repo = repo_url or (ENABLED_REPOS[0]['url'] if ENABLED_REPOS else 'https://github.com/CERN/TIGRE')
    print('Testing repo:', target_repo)
    print('Testing property:', property_name)

    result, seconds = timed_call(
        process_repository,
        repo_url=target_repo,
        provider=PROVIDER,
        model=MODEL_NAME,
        base_url=BASE_URL,
        top_k=TOP_K,
        properties=[property_name],
    )
    print(f'Elapsed time: {seconds:.3f} seconds')
    print(json.dumps(result, indent=2, ensure_ascii=False)[:4000])
    return result, seconds

In [150]:
# Property test: license
license_result, license_seconds = run_property_test('license')

Testing repo: https://github.com/cuilimeng/dEFEND-web.git
Testing property: license
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'
Elapsed time: 2.170 seconds
{
  "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
  "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
  "properties": {
    "license": {
      "value": null,
      "evidence": null,
      "confidence": 0.0
    }
  }
}


In [151]:
# Property test: installation
installation_result, installation_seconds = run_property_test('installation')

Testing repo: https://github.com/cuilimeng/dEFEND-web.git
Testing property: installation
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'
Elapsed time: 2.532 seconds
{
  "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
  "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
  "properties": {
    "installation": {
      "value": "Run the application using 'python application.py' after replacing the API key.",
      "evidence": "python application.py",
      "confidence": 0.9
    }
  }
}


In [152]:
# Property test: contact
contact_result, contact_seconds = run_property_test('contact')

Testing repo: https://github.com/cuilimeng/dEFEND-web.git
Testing property: contact
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'
Elapsed time: 2.145 seconds
{
  "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
  "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
  "properties": {
    "contact": {
      "value": null,
      "evidence": null,
      "confidence": 1.0
    }
  }
}


In [153]:
# Property test: contributors
contributors_result, contributors_seconds = run_property_test('contributors')

Testing repo: https://github.com/cuilimeng/dEFEND-web.git
Testing property: contributors
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'
Elapsed time: 1.531 seconds
{
  "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
  "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
  "properties": {
    "contributors": {
      "value": [
        {
          "name": "inproceedings",
          "github_url": "https://github.com/inproceedings"
        }
      ],
      "evidence": "Found GitHub handles/links in README.",
      "confidence": 0.95
    }
  }
}


In [154]:
# Property test: links
links_result, links_seconds = run_property_test('links')

Testing repo: https://github.com/cuilimeng/dEFEND-web.git
Testing property: links
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'
Elapsed time: 8.626 seconds
{
  "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
  "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
  "properties": {
    "links": {
      "value": [
        {
          "title": "dEFEND: A System for Explainable Fake News Detection",
          "url": null,
          "relevance": "paper",
          "is_working": null,
          "status_code": null
        },
        {
          "title": "defend: Explainable fake news detection",
          "url": null,
          "relevance": "paper",
          "is_working": null,
          "status_code": null
        }
      ],
      "evidence": "If you use the code, we appreciate it if you cite an appropriate subset of the following papers:\n\n~~~~\n@inproceedings{cui2019defend,\n  title={dEFEND: 

In [155]:
# Property test: description
description_result, description_seconds = run_property_test('description')

Testing repo: https://github.com/cuilimeng/dEFEND-web.git
Testing property: description
[Embeddings] disabled, lexical fallback only: 'BertConfig' object has no attribute 'is_decoder'
Elapsed time: 3.647 seconds
{
  "repo_url": "https://github.com/cuilimeng/dEFEND-web.git",
  "readme_url": "https://raw.githubusercontent.com/cuilimeng/dEFEND-web/master/README.md",
  "properties": {
    "description": {
      "value": "dEFEND is an explainable fake news detection tool which can exploit both news contents and user comments to jointly capture explainable top-_k_ check-worthy sentences and user comments for fake news detection.",
      "evidence": "dEFEND is an explainable fake news detection tool which can exploit both news contents and user comments to jointly capture explainable top-_k_ check-worthy sentences and user comments for fake news detection.",
      "confidence": 1.0
    }
  }
}
